# Document Store

Esta documentação detalha a classe `DocumentStore`, um componente essencial para a persistência de dados em sua arquitetura, projetado para oferecer flexibilidade e desacoplamento.

---

## Visão Geral

A classe `DocumentStore` atua como um **Roteador (Router)** e **Proxy** para operações em bancos de dados NoSQL. Seu principal objetivo é abstrair a complexidade de qual banco de dados está sendo utilizado (MongoDB ou JSON Local).

Isso permite que os desenvolvedores escrevam o código de lógica de negócio uma única vez, podendo alternar o armazenamento apenas alterando uma variável de ambiente ou um parâmetro inicial, o que é ideal para alternar entre ambientes de **Produção** (MongoDB) e **Desenvolvimento/Testes** (Local).

---

## Fluxo de Execução

O funcionamento da classe segue uma lógica de três etapas simples:

1. **Instanciação:** Ao criar o objeto, o código verifica o argumento `backend` ou a variável de ambiente `NOSQL_BACKEND`.
2. **Fábrica (Factory):** O método interno `_initialize_manager` decide qual classe gerenciadora instanciar (`MongoDBManager` ou `LocalManager`).
3. **Delegação (Proxy):** Quando um método como `save_payload` é chamado, a `DocumentStore` não processa o dado diretamente; ela "passa a bola" para o gerenciador instanciado na etapa 2.

---

## Tabela de Métodos

| **Método** | **Tipo** | **Descrição Breve** |
| --- | --- | --- |
| `__init__` | Construtor | Define o backend e inicializa o gerenciador. |
| `_initialize_manager` | Privado | Lógica de decisão (Factory) para escolher o backend. |
| `save_payload` | Proxy | Insere novos documentos no banco. |
| `fetch_documents` | Proxy | Busca e recupera documentos filtrados. |
| `update_documents` | Proxy | Atualiza dados de documentos existentes. |
| `delete_documents` | Proxy | Remove documentos da base de dados. |

---

## Arquitetura e Insights

- **Padrão Proxy:** A classe serve como uma interface unificada. O código que chama a `DocumentStore` não precisa saber *como* o MongoDB funciona, apenas *o que* quer fazer.
- **Padrão Factory:** A lógica de criação do objeto de banco de dados está centralizada em um único ponto, facilitando a adição de novos backends no futuro (como um Redis ou DynamoDB).
- **Desacoplamento:** O uso de `args` e `*kwargs` nos métodos permite que a interface seja genérica o suficiente para aceitar diferentes parâmetros exigidos por backends distintos sem quebrar a assinatura dos métodos.

---

## Detalhamento da Classe

## Classe DocumentStore

**Descrição**

Uma classe de abstração de alto nível que gerencia a persistência de documentos NoSQL. Ela centraliza as operações de CRUD (Create, Read, Update, Delete) e delega a execução para backends específicos dependendo da configuração.

**Argumentos**

- `backend` (Optional[str]): Uma string indicando o backend desejado. Valores aceitos: `"mongo"` ou `"local"`. Se `None`, busca na variável de ambiente `NOSQL_BACKEND`.

---

## Métodos

## 1. _initialize_manager

**Descrição**

Método interno que decide qual implementação de banco de dados será carregada na memória.

**Argumentos**

- Não possui argumentos diretos (utiliza o estado de `self.backend`).

**Retornos**

- `Union[MongoDBManager, LocalManager]`: Uma instância da classe responsável pela comunicação real com o banco.

**Raises**

- `ValueError`: Se o backend informado não for "mongo" ou "local".

---

## 2. save_payload

**Descrição**

Realiza a inserção de um documento ou carga de dados no backend selecionado.

**Argumentos**

- `args` / `*kwargs`: Geralmente incluem o nome do banco, nome da coleção/tabela e o dicionário de dados (payload).

**Retornos**

- `dict`: Informações sobre o sucesso da operação (ex: ID do documento inserido).

**Raises**

- `Exception`: Propaga erros específicos do driver de conexão do banco.

**Exemplos**

```bash
store = DocumentStore(backend="local")
store.save_payload(db="app", collection="logs", data={"status": "sucesso"})
```

---

## 3. fetch_documents

**Descrição**

Consulta a base de dados para retornar documentos que correspondam aos critérios informados.

**Argumentos**

- `args` / `*kwargs`: Filtros de busca (ex: `{"id": 123}`).

**Retornos**

- `list`: Uma lista contendo os documentos encontrados (dicionários).

**Exemplos**

```bash
# Busca todos os usuários com nome Enzo
users = store.fetch_documents("mydb", "users", {"name": "Enzo"})
```

---

## 4. update_documents

**Descrição**

Modifica documentos existentes na base de dados com base em um critério de seleção.

**Argumentos**

- `args` / `*kwargs`: Filtro de busca e os novos dados a serem aplicados.

**Retornos**

- `dict`: Resumo da operação (quantidade de campos alterados).

**Exemplos**

```bash
store.update_documents("mydb", "users", {"name": "Enzo"}, {"active": True})
```

---

## 5. delete_documents

**Descrição**

Remove um ou mais documentos da base de dados.

**Argumentos**

- `args` / `*kwargs`: Filtros para identificar quais documentos devem ser excluídos.

**Retornos**

- `dict`: Confirmação da remoção.

**Exemplos**

```bash
store.delete_documents("mydb", "users", {"name": "Enzo"})
```
